# 🎧 AI DJ — Local Server

Run this notebook to launch the AI-DJ backend, then open the printed URL.

The whole studio (folder selection, analysis, beatmatching, effects, EQ,
rendering, download) lives in the browser UI. This notebook is just a launcher —
you can equally run `python run_server.py --open` from a terminal.

**Requirements:** the project virtualenv (`.venv`) and `ffmpeg` on PATH.
Optional model weights go in `models_weights/` (see `train_models/`).

## 1 · Environment check

In [1]:
import sys, os, shutil
# make sure we run from the project root
if os.path.basename(os.getcwd()) == 'train_models':
    os.chdir('..')
ROOT = os.getcwd(); sys.path.insert(0, ROOT)
print('project root :', ROOT)
print('python       :', sys.version.split()[0])
print('ffmpeg       :', shutil.which('ffmpeg') or 'MISSING — install it (brew install ffmpeg)')
print('rubberband   :', shutil.which('rubberband') or 'missing (falls back to librosa stretch)')
import importlib
for m in ['librosa','soundfile','pyloudnorm','fastapi','uvicorn','torch','mutagen']:
    try: importlib.import_module(m); print(f'ok  {m}')
    except Exception as e: print(f'!!  {m}: {e}')

project root : /Users/sam/Desktop/aiolearn/My_projects/AI_DJ
python       : 3.10.19
ffmpeg       : /opt/homebrew/bin/ffmpeg
rubberband   : /opt/homebrew/bin/rubberband
ok  librosa
ok  soundfile
ok  pyloudnorm
ok  fastapi
ok  uvicorn
ok  torch
ok  mutagen


## 2 · Start the server (background thread)

Runs uvicorn in a daemon thread so the notebook stays responsive. Re-run the
cell after code changes, or use `python run_server.py --reload` for auto-reload.

In [2]:
import threading, time, uvicorn, socket
from app.api import app

def _free_port(start=5090):
    for p in range(start, start+50):
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', p)) != 0:
                return p
    return start

PORT = _free_port(5090)
config = uvicorn.Config(app, host='127.0.0.1', port=PORT, log_level='info')
server = uvicorn.Server(config)
t = threading.Thread(target=server.run, daemon=True); t.start()
time.sleep(1.5)
print(f'\n  ▶  AI DJ is live at  http://127.0.0.1:{PORT}\n')
try:
    import webbrowser; webbrowser.open(f'http://127.0.0.1:{PORT}')
except Exception: pass

INFO:     Started server process [8776]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5090 (Press CTRL+C to quit)



  ▶  AI DJ is live at  http://127.0.0.1:5090



## 3 · (Optional) Render a mix headless

Prefer the browser UI, but you can also drive the whole pipeline from Python —
handy for scripting or debugging. Point `FOLDER` at a directory of audio files.

In [3]:
from app.config import MixSettings
from app.pipeline import run_mix_job
from app.progress import PROGRESS

FOLDER = os.path.join(ROOT, 'datasets', '_demo_audio')  # <-- change me
settings = MixSettings.from_dict({
    'energy_curve': 'rising',
    'transition_intensity': 0.6,
    'effect_intensity': 0.5,
    'harmonic_priority': 0.6,
    'preserve_quality': True,
})
if os.path.isdir(FOLDER):
    PROGRESS.start('nb')
    result = run_mix_job(FOLDER, settings, output_stub='notebook_mix')
    print(result.get('wav'), '| checks passed:', result.get('checks',{}).get('passed'))
else:
    print('Set FOLDER to a real music directory to try the headless render.')

00:35:16 | INFO    | aidj | Vibe model: loaded trained GenreCNN (model_1_genre_cnn.pt) on mps
00:35:16 | INFO    | aidj | Mood model: loaded MoodTaggerNet (50 tags) on mps
00:35:16 | INFO    | aidj | Dance model: loaded GenreCNN (11 styles) on mps
00:35:16 | INFO    | aidj | active models -> vibe=cnn  mood=loaded  dance=loaded
00:35:16 | INFO    | aidj | Scanned /Users/sam/Desktop/aiolearn/My_projects/AI_DJ/datasets/_demo_audio: 4 candidate files, 4 usable
00:35:16 | INFO    | aidj | cache hit: 01 - House A - Am.wav
00:35:16 | INFO    | aidj | cache hit: 02 - House B - C.wav
00:35:16 | INFO    | aidj | cache hit: 03 - Deep C - B.wav
00:35:16 | INFO    | aidj | cache hit: 04 - Peak D - D.wav
00:35:16 | INFO    | aidj | planned 4-track set (~1.7 min, rising energy)
00:35:16 | INFO    | aidj | rendering 4 tracks -> ~1.0 min mix


INFO:     127.0.0.1:64533 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64533 - "GET /api/health HTTP/1.1" 200 OK


00:35:18 | INFO    | aidj | job complete in 2.1s -> /Users/sam/Desktop/aiolearn/My_projects/AI_DJ/outputs/notebook_mix.wav


/Users/sam/Desktop/aiolearn/My_projects/AI_DJ/outputs/notebook_mix.wav | checks passed: True


## 4 · Stop the server

The server runs in a daemon thread, so it stops when the kernel shuts down.
To stop it explicitly, run the cell below.

In [4]:
# try:
#     server.should_exit = True
#     print('server signalled to stop')
# except NameError:
#     print('server not running')

INFO:     127.0.0.1:64535 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64535 - "GET /api/health HTTP/1.1" 200 OK
INFO:     127.0.0.1:64649 - "POST /api/pick-folder HTTP/1.1" 200 OK


00:35:48 | INFO    | aidj | Scanned /Users/sam/Documents/autobahn3: 14 candidate files, 14 usable


INFO:     127.0.0.1:64649 - "POST /api/scan HTTP/1.1" 200 OK
INFO:     127.0.0.1:65301 - "POST /api/generate HTTP/1.1" 200 OK


00:36:12 | INFO    | aidj | active models -> vibe=cnn  mood=loaded  dance=loaded
INFO:     127.0.0.1:65302 - "WebSocket /ws/progress" [accepted]
INFO:     connection open
00:36:12 | INFO    | aidj | Scanned /Users/sam/Documents/autobahn3: 14 candidate files, 14 usable
00:36:12 | INFO    | aidj | analyzing: Alibi (with Pabllo Vittar & Yseult).mp3
00:36:24 | INFO    | aidj | analyzing: Don Omar, Lucenzo - Danza Kuduro.mp3
00:36:36 | INFO    | aidj | analyzing: LUZ ROJA.mp3
00:36:43 | INFO    | aidj | analyzing: MATUSHKA ULTRAFUNK.mp3
00:36:53 | INFO    | aidj | analyzing: Meet you at the Graveyard.mp3
00:37:03 | INFO    | aidj | analyzing: Mortals Funk Remix.mp3
00:37:12 | INFO    | aidj | analyzing: Royalty.mp3
00:37:26 | INFO    | aidj | analyzing: SENTE MAIS.mp3
00:37:33 | INFO    | aidj | analyzing: SLAVA POWER!.mp3
00:37:41 | INFO    | aidj | analyzing: Shakira - Dai Dai (Ft Burna Boy).mp3
00:37:55 | INFO    | aidj | analyzing: Tayna - Si Ai.mp3
00:38:05 | INFO    | aidj | analyzing